# Solomonoff Gap Benchmark — Week 1 Kaggle T4 Inference

**Run this notebook on Kaggle with GPU T4 x1.**

Settings → Accelerator: GPU T4 x1

## What This Notebook Does
1. Installs dependencies
2. Loads `data/sequences_mvp.json` (300 sequences)
3. Validates tokenizer mapping (5-prompt gate)
4. Runs the full EL_gzip benchmark for 2 local models
5. Saves `results/el_gzip_results.csv` with checkpoints every 50 sequences

## Week 1 Models
- Primary: `microsoft/Phi-3-mini-4k-instruct` (ungated, ~3.8B)
- Primary: `meta-llama/Llama-3.2-3B-Instruct` (needs HF token + accepted license)
- Fallback: `TinyLlama/TinyLlama-1.1B-Chat-v1.0` (ungated, very small)

## Hard Rules
- Do NOT use `model.generate(output_scores=True)` — use direct logits
- Save invalid-token mass BEFORE renormalization
- Run the 5-prompt toy gate before the full 300-sequence run
- Save checkpoints every 50 sequences

In [ ]:
# Cell 1 — Install dependencies
!pip install transformers accelerate bitsandbytes -q

In [ ]:
# Cell 2 — HuggingFace authentication (only needed for gated models like Llama 3.2)
import os
import huggingface_hub

# In Kaggle: Add Secrets → HF_TOKEN (from huggingface.co/settings/tokens)
hf_token = os.environ.get('HF_TOKEN', None)
if hf_token:
    huggingface_hub.login(token=hf_token)
    print('Logged in to HuggingFace')
else:
    print('No HF_TOKEN found — using ungated models only')

In [ ]:
# Cell 3 — Setup paths and install package
import sys
from pathlib import Path

# If running on Kaggle, clone the repo first:
# !git clone https://github.com/ajinkya-awari/solomonoff-bench.git
# repo_root = Path('/kaggle/working/solomonoff-bench')

# If running locally:
repo_root = Path('/kaggle/working/solomonoff-bench')
sys.path.insert(0, str(repo_root / 'src'))

dataset_path = repo_root / 'data' / 'sequences_mvp.json'
results_dir = repo_root / 'results'
results_dir.mkdir(exist_ok=True)
(results_dir / 'figures').mkdir(exist_ok=True)

print('Dataset path:', dataset_path)
print('Results dir:', results_dir)

In [ ]:
# Cell 4 — Verify GPU
import torch
print('CUDA available:', torch.cuda.is_available())
if torch.cuda.is_available():
    print('Device:', torch.cuda.get_device_name(0))
    print('VRAM:', round(torch.cuda.get_device_properties(0).total_memory / 1e9, 1), 'GB')

In [ ]:
# Cell 5 — Load dataset and inspect
import json

with open(dataset_path) as f:
    dataset = json.load(f)

records = dataset['records']
print('Total sequences:', len(records))
print('Complexity levels:', dataset['complexity_levels'])
print()
for level in sorted(set(r['complexity_level'] for r in records)):
    lvl_recs = [r for r in records if r['complexity_level'] == level]
    print('  Level', level, ':', len(lvl_recs), 'sequences,',
          'program_bits =', lvl_recs[0]['program_bits'])

In [ ]:
# Cell 6 — Model selection
# Uncomment the model you want to run.

# Option A: Phi-3 Mini (ungated, recommended starting point)
MODEL_NAME = 'microsoft/Phi-3-mini-4k-instruct'

# Option B: Llama 3.2 3B (requires accepted license + HF_TOKEN)
# MODEL_NAME = 'meta-llama/Llama-3.2-3B-Instruct'

# Option C: TinyLlama (ungated fallback, very small)
# MODEL_NAME = 'TinyLlama/TinyLlama-1.1B-Chat-v1.0'

# Use 4-bit quantization on T4 (saves VRAM for larger models)
LOAD_IN_4BIT = True

print('Running benchmark for:', MODEL_NAME)
print('4-bit quantization:', LOAD_IN_4BIT)

In [ ]:
# Cell 7 — Build tokenizer map and run 5-prompt toy gate
# This MUST pass before the full 300-sequence run.

from transformers import AutoTokenizer
from solomonoff_bench.models.tokenizer_validation import build_binary_token_map
import torch
import torch.nn.functional as F

tokenizer = AutoTokenizer.from_pretrained(MODEL_NAME, trust_remote_code=True)
token_map = build_binary_token_map(tokenizer, MODEL_NAME)

print('=== Tokenizer Validation Table ===')
print(token_map.summary())
print()
print('Valid zero tokens:', token_map.zero_ids)
print('Valid one tokens:', token_map.one_ids)
print('Is valid:', token_map.is_valid())

In [ ]:
# Cell 8 — Load model and run 5-prompt toy validation gate

from solomonoff_bench.models.local_model import LocalHFScorer

scorer = LocalHFScorer(
    model_name_or_path=MODEL_NAME,
    load_in_4bit=LOAD_IN_4BIT,
)

# Pick 5 sequences for the toy gate
toy_seqs = [r['sequence'] for r in records[:5]]

gate_result = scorer.validate_toy_sequences(toy_seqs)
print('=== Toy Gate Result ===')
print('Gate passed:', gate_result['gate_passed'])
print('Mean valid binary mass:', round(gate_result['mean_valid_binary_mass'], 4))
print('Mean invalid mass:', round(gate_result['mean_invalid_mass'], 4))
print()
if not gate_result['gate_passed']:
    raise RuntimeError('Tokenizer validation gate FAILED. Do not run the full benchmark.')
print('Gate passed — proceeding to full benchmark.')

In [ ]:
# Cell 9 — Full benchmark run (300 sequences, checkpointing every 50)

import csv
import json
import time
from solomonoff_bench.baselines.gzip_baseline import GzipBaseline
from solomonoff_bench.metrics.excess_loss import compute_el_gzip

CONTEXT_LEN = 100
PREDICT_LEN = 100
CHECKPOINT_EVERY = 50

model_slug = MODEL_NAME.replace('/', '--')
csv_path = results_dir / ('el_gzip_' + model_slug + '.csv')
log_path = results_dir / 'benchmark_log.jsonl'

# Load completed IDs for resumability
completed_ids = set()
if log_path.exists():
    with open(log_path) as f:
        for line in f:
            try:
                entry = json.loads(line.strip())
                if entry.get('status') == 'done':
                    completed_ids.add(entry['sequence_id'])
            except Exception:
                pass

print('Already scored:', len(completed_ids), '— resuming')

gzip_baseline = GzipBaseline()
rows = []
fieldnames = [
    'sequence_id', 'complexity_level', 'n_states', 'program_bits',
    'model', 'context_len', 'predict_len',
    'h_model_bits_per_sym', 'h_gzip_bits_per_sym', 'h_gzip_is_negative',
    'el_gzip', 'mean_invalid_mass', 'mean_valid_binary_mass',
]

write_header = not csv_path.exists()
csv_file = open(csv_path, 'a', newline='', encoding='utf-8')
writer = csv.DictWriter(csv_file, fieldnames=fieldnames)
if write_header:
    writer.writeheader()

log_file = open(log_path, 'a', encoding='utf-8')

done = 0
t0 = time.time()

for rec in records:
    sid = rec['sequence_id']
    if sid in completed_ids:
        continue

    seq = rec['sequence']
    window = scorer.score_sequence_window(seq, CONTEXT_LEN, PREDICT_LEN)
    p0_list = [r['p0_renorm'] for r in window]
    mean_invalid = sum(r['invalid_mass'] for r in window) / len(window)
    mean_valid = sum(r['valid_binary_mass'] for r in window) / len(window)

    el = compute_el_gzip(seq, p0_list, CONTEXT_LEN, PREDICT_LEN)
    gzip_baseline.score(seq[:CONTEXT_LEN], seq[CONTEXT_LEN:])  # track negative deltas

    row = {
        'sequence_id': sid,
        'complexity_level': rec['complexity_level'],
        'n_states': rec['n_states'],
        'program_bits': rec['program_bits'],
        'model': MODEL_NAME,
        'context_len': CONTEXT_LEN,
        'predict_len': PREDICT_LEN,
        'h_model_bits_per_sym': round(el['h_model_bits_per_sym'], 6),
        'h_gzip_bits_per_sym': round(el['h_gzip_bits_per_sym'], 6),
        'h_gzip_is_negative': el['h_gzip_is_negative'],
        'el_gzip': round(el['el_gzip'], 6),
        'mean_invalid_mass': round(mean_invalid, 6),
        'mean_valid_binary_mass': round(mean_valid, 6),
    }
    writer.writerow(row)
    csv_file.flush()

    log_file.write(json.dumps({
        'status': 'done', 'sequence_id': sid, 'model': MODEL_NAME,
        'el_gzip': el['el_gzip'], 'h_gzip_is_negative': el['h_gzip_is_negative'],
    }) + '\n')
    log_file.flush()

    done += 1
    if done % CHECKPOINT_EVERY == 0:
        elapsed = time.time() - t0
        rate = done / elapsed
        remaining = (len(records) - done) / max(rate, 1e-6)
        print('Checkpoint:', done, '/', len(records),
              '| elapsed', round(elapsed, 0), 's',
              '| ETA', round(remaining / 60, 1), 'min')

csv_file.close()
log_file.close()

print('\n=== Benchmark Complete ===')
print('Scored:', done, 'sequences')
print('Results saved to:', csv_path)
print()
print('Gzip diagnostics:')
diag = gzip_baseline.diagnostics()
print('  Total calls:', diag['total_calls'])
print('  Negative delta count:', diag['negative_delta_count'])
print('  Negative delta rate:', round(diag['negative_delta_rate'], 3))

# Save gzip diagnostics to log
with open(log_path, 'a') as f:
    f.write(json.dumps({'event': 'gzip_diagnostics', 'model': MODEL_NAME, **diag}) + '\n')

In [ ]:
# Cell 10 — Quick results preview
import pandas as pd

df = pd.read_csv(csv_path)
print('Results shape:', df.shape)
print()
print('EL_gzip by complexity level:')
print(df.groupby('complexity_level')[['el_gzip', 'h_model_bits_per_sym', 'h_gzip_bits_per_sym']].mean().round(4))
print()
print('Tokenizer health:')
print('  Mean invalid mass:', df['mean_invalid_mass'].mean().round(4))
print('  Mean valid binary mass:', df['mean_valid_binary_mass'].mean().round(4))